# RailGuard AI â€” Notebook 04: Railway Defect Detection Inference Demo

This notebook demonstrates interactive object detection using the trained **YOLO11 Nano model weights** (`ai-service/runs/yolo11_railguard_v1/weights/best.pt`).

### Objectives:
1. Load trained YOLO11 weights using Ultralytics `YOLO` API.
2. Perform inference on unseen railway inspection images from the **Test split** (`dataset/railway-defect-dataset/images/test/`).
3. Render bounding boxes with defect class labels, confidence scores, and pixel coordinates.
4. Format prediction metadata into structured JSON / Pandas DataFrames.

In [1]:
import os
import glob
import cv2
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

# Define paths
WEIGHTS_PATH = os.path.abspath("../ai-service/runs/yolo11_railguard_v1/weights/best.pt")
TEST_IMG_DIR = os.path.abspath("../dataset/railway-defect-dataset/images/test")

print(f"Weights Path: {WEIGHTS_PATH}")
print(f"Weights Exists: {os.path.exists(WEIGHTS_PATH)}")
print(f"Test Directory Exists: {os.path.exists(TEST_IMG_DIR)}")

Weights Path: C:\Users\Ankadala Triveni\OneDrive\Documents\RailGuardAI\ai-service\runs\yolo11_railguard_v1\weights\best.pt
Weights Exists: True
Test Directory Exists: True


## 1. Load Trained YOLO11 Model

In [2]:
if os.path.exists(WEIGHTS_PATH):
    model = YOLO(WEIGHTS_PATH)
    print("Model loaded successfully!")
    print(f"Target Classes ({len(model.names)}): {model.names}")
else:
    print("Model checkpoint best.pt not found yet. It will be saved upon completion of training.")

Model loaded successfully!
Target Classes (15): ['broken_rail_high', 'broken_rail_low', 'broken_rail_medium', 'corrosion_high', 'corrosion_low', 'corrosion_medium', 'head_checks_high', 'head_checks_low', 'head_checks_medium', 'railway_join_bar_defects_high', 'railway_join_bar_defects_low', 'railway_join_bar_defects_medium', 'squats_high', 'squats_low', 'squats_medium']


## 2. Execute Inference on Unseen Test Split Images

In [3]:
if 'model' in locals():
    test_images = glob.glob(os.path.join(TEST_IMG_DIR, "*.jpg"))[:6]
    results_list = []
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    for idx, img_path in enumerate(test_images):
        results = model.predict(source=img_path, conf=0.25, imgsz=640, verbose=False)[0]
        
        # Plotted BGR image from Ultralytics
        res_bgr = results.plot()
        res_rgb = cv2.cvtColor(res_bgr, cv2.COLOR_BGR2RGB)
        
        # Extract detection boxes metadata
        for box in results.boxes:
            cls_id = int(box.cls[0].item())
            cls_name = model.names[cls_id]
            conf = float(box.conf[0].item())
            xyxy = box.xyxy[0].tolist()
            
            results_list.append({
                "Image": os.path.basename(img_path),
                "Defect Class": cls_name,
                "Confidence": round(conf, 4),
                "BBox (x1, y1, x2, y2)": [round(v, 1) for v in xyxy]
            })
            
        ax = axes[idx // 3, idx % 3]
        ax.imshow(res_rgb)
        ax.set_title(f"{os.path.basename(img_path)} ({len(results.boxes)} defects)", fontsize=11)
        ax.axis("off")
        
    plt.suptitle("RailGuard AI â€” Real-Time Inference on Unseen Test Split", fontsize=15, fontweight="bold")
    plt.tight_layout()
    plt.show()
    
    print("\n=== Detected Defect Bounding Boxes ===")
    df_detections = pd.DataFrame(results_list)
    if not df_detections.empty:
        print(df_detections.to_string(index=False))
    else:
        print("No defect bounding boxes exceeded confidence threshold 0.25 on sample images.")


=== Detected Defect Bounding Boxes ===
                                                                 Image                 Defect Class  Confidence        BBox (x1, y1, x2, y2)
1-MOV_20201221091849_0313_JPEG.rf.48eef42909ba63d6e55e811a937455dd.jpg railway_join_bar_defects_low      0.5603 [259.4, 272.1, 383.7, 385.7]
1-MOV_20201221091849_2632_JPEG.rf.22516a0a8d950d6ef031a9ee96918388.jpg                   squats_low      0.7855   [331.0, 0.1, 387.5, 119.1]
1-MOV_20201221091849_4217_JPEG.rf.02b23db3b9a28169e9652eba7dce404a.jpg                   squats_low      0.1726 [316.6, 183.4, 375.0, 277.2]
1-MOV_20201221091849_4481_JPEG.rf.7f2ffde12342312679344dc59af76ad2.jpg              head_checks_low      0.2011 [363.5, 380.2, 384.5, 422.5]
1-MOV_20201221091849_4626_JPEG.rf.2f427635743b841650edcaf4210d98e9.jpg              head_checks_low      0.5154 [328.8, 273.8, 375.3, 327.3]
